# v2 — semantic retrieval, polysemy, and ensemble fusion

[![Python 3.11](https://img.shields.io/badge/python-3.11-blue.svg)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/notebook-jupyter-orange.svg)](https://jupyter.org/)

---

## What this notebook does

The v1 pipeline (companion repository
[`youtube-protest-tags`](https://github.com/i-sulzhi/youtube-protest-tags))
treats videos exclusively through their tags and a tag co-occurrence
network. The accompanying EDA notebook (`notebooks/EDA.ipynb`) showed
that **the protest keyword appears in only 9.4% of the politics-topic
subset** — meaning 90.6% of politically relevant content is invisible
to a tag- or keyword-based retriever.

This notebook documents the v2 pipeline, which addresses that gap by
embedding video metadata into a multilingual semantic space and then
combining results across two encoders.

- **Stage A** — encode `TITLE + DESCRIPTION` for the 21,710 text-ready
  videos with both `multilingual-e5-small` (384-dim) and
  `multilingual-e5-large` (1024-dim), persist to LanceDB.
- **Stage B** — retrieve top-100 videos per natural-language anchor query
  with each encoder. Quantify the recall gain over the regex-keyword
  baseline.
- **Stage C** — cluster the embedding-space positions of every
  protest-mentioning video. Demonstrate that polysemy structure
  reproduces between encoders (evidence that the structure is real, not
  an encoder artefact).
- **Stage D** — fuse the two rankers' results (RRF, CombSUM, UNION).
  Quantify the further recall gain from ensembling.

> **Encoder rationale.** `multilingual-e5-small` (118M params) runs at
> ~25 vec/s on M2 16 GB and finishes the corpus in 6:34. `multilingual-e5-large`
> (560M params) requires `--batch 8` to avoid thermal throttling on the
> same hardware, and finishes in 32:01. Both fit comfortably in 16 GB
> with fp16 weights.

## Table of contents

1. [Setup](#1-setup)
2. [Stage A: encoder comparison](#2-stage-a)
3. [Stage B: anchor retrieval per encoder](#3-stage-b)
4. [Stage B: recall gain over the regex baseline](#4-recall-gain)
5. [Stage B: top-100 overlap between encoders](#5-overlap)
6. [Stage C: polysemy via embedding clusters](#6-stage-c)
7. [Stage C: cluster stability across encoders](#7-cluster-stability)
8. [Stage D: ensemble fusion](#8-stage-d)
9. [Stage D: comparison of fusion strategies](#9-fusion-comparison)
10. [Findings and next steps](#10-findings)


<a id="1-setup"></a>
## 1. Setup

This notebook expects artefacts produced by the four pipeline stage
scripts:

```
outputs/v2_lancedb/videos.lance/         (e5-small, Stage A)
outputs/v2_lancedb/videos_e5large.lance/ (e5-large, Stage A)
outputs/retrieval/                       (e5-small, Stage B)
outputs/retrieval_e5large/               (e5-large, Stage B)
outputs/polysemy/                        (e5-small, Stage C)
outputs/polysemy_e5large/                (e5-large, Stage C)
outputs/retrieval_ensemble/              (Stage D, three fusion strategies)
```

Run the four scripts in order from the project root before executing
this notebook (`scripts/overnight_chain.sh` orchestrates the e5-large
run; see `README.md` for the full invocation).


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lancedb
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# Project root holds the outputs/ directory. The notebook lives in
# notebooks/, so it works whether Jupyter is launched from the project
# root (cwd is the root) or from inside notebooks/ (we step up one
# level). Override with PROTEST_ROOT if needed.
def _find_root() -> str:
    env = os.environ.get("PROTEST_ROOT")
    if env:
        return env
    here = Path.cwd()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "outputs").exists():
            return str(candidate)
    return "."

ROOT      = _find_root()
LANCE_DIR = os.path.join(ROOT, "outputs/v2_lancedb")
print(f"ROOT={ROOT}")


<a id="2-stage-a"></a>
## 2. Stage A — encoder comparison

Both encoders embed the same input (`TITLE + " . " + DESCRIPTION`,
URL-replaced, truncated to 2000 chars, prefixed with `passage: ` per the
e5 convention). The output is a LanceDB table with one row per video and
the original metadata (country combo, language, publication date,
protest-keyword flags) for filtering at retrieval time.


In [ ]:
db = lancedb.connect(LANCE_DIR)

tbl_small = db.open_table("videos")
tbl_large = db.open_table("videos_e5large")

print(f"e5-small  rows={len(tbl_small):,}  dim={tbl_small.schema.field('vector').type.list_size}")
print(f"e5-large  rows={len(tbl_large):,}  dim={tbl_large.schema.field('vector').type.list_size}")

# Confirm both tables describe the same set of videos
ids_small = set(tbl_small.search().select(['YT_ID']).limit(len(tbl_small)).to_pandas()['YT_ID'])
ids_large = set(tbl_large.search().select(['YT_ID']).limit(len(tbl_large)).to_pandas()['YT_ID'])
assert ids_small == ids_large, "encoders embed different video sets"
print(f"Both tables cover the same {len(ids_small):,} videos.")


In [ ]:
# Encoder cost / quality summary (measured)
encoder_summary = pd.DataFrame([
    {"model": "multilingual-e5-small", "params_M": 118, "dim": 384, "batch": 32,
     "vec_per_sec": 25, "wall_time_min": 6.6, "lancedb_MB": 95},
    {"model": "multilingual-e5-large", "params_M": 560, "dim": 1024, "batch": 8,
     "vec_per_sec": 11, "wall_time_min": 32.0, "lancedb_MB": 250},
])
encoder_summary


<a id="3-stage-b"></a>
## 3. Stage B — anchor retrieval per encoder

Seven Russian-language anchor queries cover the principal political
contexts in the corpus: Belarusian protests 2020, Russian anti-war
actions 2022, the Navalny mobilisations, the autumn-2022 mobilisation
protests, the Euromaidan revolution 2013–2014, current Ukrainian
protest, and a generic protest-action prompt. For each anchor we
retrieve the top-100 videos by cosine similarity in the encoder's
embedding space.


In [ ]:
hits_small = pd.read_csv(os.path.join(ROOT, "outputs/retrieval/topk_per_anchor.csv"))
hits_large = pd.read_csv(os.path.join(ROOT, "outputs/retrieval_e5large/topk_per_anchor.csv"))

# Per-anchor top similarity comparison
def top_sim_summary(df, name):
    g = df.groupby("anchor")["similarity"].agg(top1="max", mean="mean").round(3)
    g.columns = pd.MultiIndex.from_product([[name], g.columns])
    return g

side = pd.concat([top_sim_summary(hits_small, "e5-small"),
                  top_sim_summary(hits_large, "e5-large")], axis=1)
side


<a id="4-recall-gain"></a>
## 4. Stage B — recall gain over the regex baseline

The headline question for v2 is: *how many videos does
embedding-retrieval find that the protest-keyword regex would miss?*
For each anchor, we report the share of top-100 hits that contain no
surface protest keyword anywhere — the "newly surfaced" videos. A
high newly-surfaced rate indicates the encoder is recovering content
that names protest by metonymy (event names, people, slogans) rather
than by the literal token *protest*.


In [ ]:
def newly_pct(df, anchor):
    sub = df[df["anchor"] == anchor]
    n_newly = (~sub["protest_anywhere"].astype(bool)).sum()
    return 100 * n_newly / len(sub) if len(sub) else 0

anchors = sorted(hits_small["anchor"].unique())
recall = pd.DataFrame({
    "anchor":   anchors,
    "small":    [newly_pct(hits_small, a) for a in anchors],
    "large":    [newly_pct(hits_large, a) for a in anchors],
}).round(1)
recall


In [ ]:
# Visualise per-anchor recall gain
plot_df = recall.melt(id_vars="anchor", var_name="encoder", value_name="newly_pct")
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=plot_df, y="anchor", x="newly_pct", hue="encoder",
            palette={"small": "#3a6ea5", "large": "#5a8e3a"}, ax=ax)
ax.set_xlabel("% of top-100 with no surface protest keyword")
ax.set_title("Recall gain over the regex baseline (per anchor, per encoder)")
plt.tight_layout(); plt.show()


On six of seven anchors the recall gain is substantial in both encoders,
and on three (RU-mobilisation, RU-antiwar, general-protest-action)
e5-large recovers 11–18 percentage points more newly-surfaced content
than e5-small. The pattern is consistent with what we expect: e5-large
maps complex multi-clause political concepts more reliably than e5-small,
particularly when the surface vocabulary differs from the query language.


<a id="5-overlap"></a>
## 5. Stage B — top-100 overlap between encoders

The two encoders have different score distributions and pick different
*best matches* from the same corpus. Quantifying the per-anchor overlap
of their top-100 lists tells us whether one model is a strict superset
of the other (in which case the smaller one is redundant) or whether
they recover complementary slices (in which case fusion will help).


In [ ]:
rows = []
for a in anchors:
    s = set(hits_small[hits_small["anchor"]==a]["YT_ID"])
    l = set(hits_large[hits_large["anchor"]==a]["YT_ID"])
    rows.append({
        "anchor": a,
        "common": len(s & l),
        "small_only": len(s - l),
        "large_only": len(l - s),
    })
overlap = pd.DataFrame(rows)
print(overlap.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 3.8))
overlap.set_index("anchor")[["common","small_only","large_only"]].plot.barh(
    stacked=True, ax=ax, color=["#888", "#3a6ea5", "#5a8e3a"], edgecolor="white")
ax.set_xlabel("videos in top-100 (composition by ranker)")
ax.set_title("Top-100 overlap between e5-small and e5-large")
ax.invert_yaxis()
plt.tight_layout(); plt.show()


Per-anchor overlap is **17 to 56 of 100**. The two encoders mostly
disagree on which 100 videos belong at the top — a strong signal that
their pickings are *complementary* and that fusion (Stage D) should
materially improve recall.


<a id="6-stage-c"></a>
## 6. Stage C — polysemy via embedding clusters

We restrict to the 2,034 protest-mentioning videos (those with a
protest-root keyword in title or description) and run KMeans (k=8) on
their embeddings. Each cluster is an inferred *sense* of the protest
term. If clusters align with country-event combinations (BY-2020,
RU-antiwar-2022, UA-Maidan, …), polysemy is demonstrated empirically
rather than asserted.


In [ ]:
sum_small = pd.read_csv(os.path.join(ROOT, "outputs/polysemy/cluster_summary.csv"))
sum_large = pd.read_csv(os.path.join(ROOT, "outputs/polysemy_e5large/cluster_summary.csv"))
print("=== e5-small ===")
print(sum_small[["cluster","n_videos","year_median","top_langs","top_country_combos"]].to_string(index=False))
print("\n=== e5-large ===")
print(sum_large[["cluster","n_videos","year_median","top_langs","top_country_combos"]].to_string(index=False))


In [ ]:
# Per-cluster country-combo composition for both encoders
def heat(path, title, ax):
    a = pd.read_csv(path)
    ct = pd.crosstab(a["cluster"], a["country_combo"])
    sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar_kws={'label':'videos'})
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
heat(os.path.join(ROOT, "outputs/polysemy/cluster_assignments.csv"),
     "e5-small clusters × country combo", axes[0])
heat(os.path.join(ROOT, "outputs/polysemy_e5large/cluster_assignments.csv"),
     "e5-large clusters × country combo", axes[1])
plt.tight_layout(); plt.show()


<a id="7-cluster-stability"></a>
## 7. Stage C — cluster stability across encoders

Comparing the two cluster summaries side by side shows that **the same
eight senses appear in both encoders**, only the cluster numbering is
different (KMeans random init). Each e5-small cluster maps one-to-one
to an e5-large cluster with the same dominant language, country combo
and year-median.


In [ ]:
# Sense-correspondence table built from the summary CSVs
sense_table = pd.DataFrame([
    {"sense": "EN Belarus 2020 (DW, AP, Reuters)",        "small": "C5 (n=231)", "large": "C5 (n=225)"},
    {"sense": "DE Belarus 2020 (ARTE, SPIEGEL, ZDF)",     "small": "C0 (n=150)", "large": "C3 (n=125)"},
    {"sense": "RU/BE Belsat-style opposition journalism", "small": "C1 (n=128)", "large": "C7 (n=129)"},
    {"sense": "RU news talk-shows on Lukashenka",         "small": "C4 (n=276)", "large": "C6 (n=278)"},
    {"sense": "EN Russia/Navalny (DW)",                   "small": "C3 (n=278)", "large": "C4 (n=286)"},
    {"sense": "RU bloggers on Navalny / war",             "small": "C2 (n=292)", "large": "C1 (n=374)"},
    {"sense": "Mixed-language war commentary (UR/U/R)",   "small": "C7 (n=487)", "large": "C2 (n=404)"},
    {"sense": "EN/DE war analysis",                       "small": "C6 (n=192)", "large": "C0 (n=213)"},
])
sense_table


In [ ]:
# Cluster purity — how concentrated is each cluster on its top country combo?
def cluster_purity(path):
    a = pd.read_csv(path)
    pur = []
    for k in sorted(a["cluster"].unique()):
        sub = a[a["cluster"] == k]
        if not len(sub): continue
        top = sub["country_combo"].value_counts().iloc[0]
        pur.append(top / len(sub))
    return pur

import statistics as st
pur_s = cluster_purity(os.path.join(ROOT, "outputs/polysemy/cluster_assignments.csv"))
pur_l = cluster_purity(os.path.join(ROOT, "outputs/polysemy_e5large/cluster_assignments.csv"))
print(f"Mean cluster purity (top combo / size):")
print(f"  e5-small: {100*st.mean(pur_s):.1f}%   range {100*min(pur_s):.1f}% – {100*max(pur_s):.1f}%")
print(f"  e5-large: {100*st.mean(pur_l):.1f}%   range {100*min(pur_l):.1f}% – {100*max(pur_l):.1f}%")


Cluster purity is **61.2% (small) vs 62.0% (large)** — essentially
identical. The polysemy structure is **encoder-stable**: both
independently find the same eight senses with the same dominant
contexts. This is the key empirical finding of Stage C — the polysemy
of «протест» is a property of the corpus, not of any one encoder.


<a id="8-stage-d"></a>
## 8. Stage D — ensemble fusion

The top-100 overlap between encoders is only 17–56 of 100. The encoders
recover complementary slices of the corpus, so combining them should
materially improve recall. Three fusion strategies are produced and
compared.

- **RRF** (Reciprocal Rank Fusion, k=60). Rank-based, robust to score
  scale differences. Each video's fused score is the sum of `1/(k + rank)`
  across the two rankers. RRF rewards consensus picks (videos in the top
  of both rankers) and is the standard precision-leaning choice.
- **CombSUM** with min-max normalised similarity. Each ranker's
  similarities are min-max scaled within an anchor, then summed.
  Videos in only one ranker get half-credit. Balanced.
- **UNION** (recall-max). Keep every video appearing in either ranker's
  top-K. Rank by best-rank-across-rankers. Maximises distinct video
  coverage; intended for downstream human review where breadth beats
  precision.


In [ ]:
ensemble_summary = pd.read_csv(os.path.join(ROOT, "outputs/retrieval_ensemble/comparison_summary.csv"))
print(ensemble_summary.to_string(index=False))


<a id="9-fusion-comparison"></a>
## 9. Stage D — comparison of fusion strategies

The recall use case for this corpus is "find as many protest-relevant
videos as possible for downstream content analysis or human review",
not "rank a small set of high-confidence answers". The relevant metric
is therefore **distinct video coverage**, not per-anchor newly-surfaced
percentage.


In [ ]:
hits_rrf  = pd.read_csv(os.path.join(ROOT, "outputs/retrieval_ensemble/topk_per_anchor.csv"))
hits_comb = pd.read_csv(os.path.join(ROOT, "outputs/retrieval_ensemble/combsum_per_anchor.csv"))
hits_uni  = pd.read_csv(os.path.join(ROOT, "outputs/retrieval_ensemble/union_per_anchor.csv"))

coverage = pd.DataFrame([
    {"strategy": "e5-small alone",        "distinct_videos": hits_small["YT_ID"].nunique()},
    {"strategy": "e5-large alone",        "distinct_videos": hits_large["YT_ID"].nunique()},
    {"strategy": "RRF (top-100/anchor)",  "distinct_videos": hits_rrf["YT_ID"].nunique()},
    {"strategy": "CombSUM (top-100/anchor)", "distinct_videos": hits_comb["YT_ID"].nunique()},
    {"strategy": "UNION (all distinct)",  "distinct_videos": hits_uni["YT_ID"].nunique()},
])
coverage


In [ ]:
# Newly-surfaced (no surface protest keyword) per strategy, deduped across anchors
def distinct_newly(df):
    df = df.copy()
    df["newly"] = ~df["protest_anywhere"].astype(bool)
    return df[df["newly"]]["YT_ID"].nunique()

newly_dedup = pd.DataFrame([
    {"strategy": "e5-small alone",  "newly_surfaced": distinct_newly(hits_small)},
    {"strategy": "e5-large alone",  "newly_surfaced": distinct_newly(hits_large)},
    {"strategy": "RRF",             "newly_surfaced": distinct_newly(hits_rrf)},
    {"strategy": "CombSUM",         "newly_surfaced": distinct_newly(hits_comb)},
    {"strategy": "UNION",           "newly_surfaced": distinct_newly(hits_uni)},
])
print(newly_dedup.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(newly_dedup["strategy"], newly_dedup["newly_surfaced"],
        color=["#3a6ea5","#5a8e3a","#a55e3a","#a55e3a","#a55e3a"], edgecolor="white")
ax.set_xlabel("Distinct newly-surfaced videos (no protest keyword in title/desc)")
ax.set_title("Recall gain across fusion strategies")
ax.invert_yaxis()
plt.tight_layout(); plt.show()


**RRF and CombSUM do not help for this recall task** — both keep top-100
per anchor, so distinct video coverage stays at ~500, the same as a single
encoder. RRF additionally *lowers* the per-anchor newly-surfaced rate
because it rewards consensus picks, which tend to share surface keywords.

**UNION is the right strategy.** It surfaces 799 distinct videos (~60% more
than either single encoder) and 527 newly-surfaced videos — versus 315 for
e5-small alone and 329 for e5-large alone. The ensemble effectively
recovers the *recall complement* of the two encoders. The trade-off is
that UNION is wider and noisier; precision is the cost.


<a id="10-findings"></a>
## 10. Findings and next steps

### 10.1 Findings

The four stages address questions that the original tag-only pipeline
could not answer.

**Encoder cost on a 16 GB M2.** Both `multilingual-e5-small` and
`multilingual-e5-large` run locally on Apple Silicon via MPS. The small
encoder finishes the 21,710-video subset in 6:34 at batch=32; the large
encoder finishes in 32:01 at batch=8 (larger batches thermal-throttle).
For an offline pipeline this is a one-time cost.

**Recall gain over regex (Stage B).** Embedding-based retrieval surfaces
a substantial share of videos that contain no surface protest keyword
but are semantically aligned with anchor queries. The exact share
depends on the anchor; the median across anchors is roughly 55%
newly-surfaced. e5-large outperforms e5-small by 11–18 percentage points
on hard anchors (mobilisation, anti-war, generic protest-action).

**Encoder complementarity (Stage B).** The two encoders' top-100 lists
overlap by only 17–56 / 100 per anchor. They find different best
matches, which motivates the ensemble in Stage D.

**Polysemy is encoder-stable (Stage C).** Both encoders independently
produce eight clusters with identical senses (Belarus 2020 in
EN/DE/RU+BE/RU-talkshow; Russia in EN/RU; war commentary in mixed
languages). Mean cluster purity is 61–62% in both. Treating «протест»
as a single concept across these eight clusters loses information that
a per-cluster view recovers.

**Ensemble fusion (Stage D).** RRF and CombSUM are precision-leaning and
do not improve recall on this corpus — they keep top-100 per anchor
and bias toward consensus picks. UNION fusion surfaces **527 distinct
newly-surfaced videos**, a +60% gain over the best single encoder.
The ensemble is the right strategy when the downstream task is
human review or content analysis on a wider candidate pool.

### 10.2 Limitations

- The text-ready subset is 21,710 videos out of the 401K corpus,
  because `VIDEO_METADATA` was captured during a single 46-day window.
  To extend to the full corpus, additional metadata fetches via the
  YouTube API are required.
- Anchor queries are written in Russian to match the corpus; an
  English-language re-formulation may surface a different slice.
- Stage C uses KMeans, which assumes spherical clusters in embedding
  space. HDBSCAN or Gaussian mixtures would be more flexible but
  less interpretable.
- No labelled validation set has been used; precision is assessed
  qualitatively from sample inspection.

### 10.3 Next steps

1. **Manual inspection of `outputs/retrieval_ensemble/newly_surfaced.csv`**
   — sample 200–300 videos stratified by country combo and anchor,
   label as protest-relevant / not, then report precision and recall
   on the labelled set.
2. **Stage E — train a classifier** on (embedding + country combo +
   tag features + channel proxy) using the labelled set; use it to
   re-rank the UNION ensemble.
3. **Channel-proxy enrichment** — cluster videos by description-head
   boilerplate (612 reusable fragments from the EDA notebook) and
   add cluster ID as a classifier feature.
4. **Anchor refinement** — narrower temporal anchors (Bolotnaya 2011,
   October-2020 women's march, autumn-2022 mobilisation) and
   anchor-language ablations.
5. **Full-corpus extension** — re-fetch title and description for the
   remaining ~380K videos via YouTube API, embed, and rerun the
   polysemy clustering on the larger sample. The shape of the
   country-combo diagonal will indicate whether the present subset's
   pattern generalises.
